# PlotProof — Farm-Gate Price Intelligence (real data, honest forecasts)

**The problem this attacks:** a middleman's margin lives on information asymmetry. A smallholder
who doesn't know this month's Colombo auction tea price cannot judge the offer made at their gate.

**What this builds:** a monthly price-intelligence artifact for the commodities in PlotProof's
catalog, from the **World Bank Commodity Price Data ("Pink Sheet")** — free, CC BY 4.0, monthly
since 1960, and it includes the literal *Tea, Colombo* series (Ceylon tea's own auction).

The pipeline: download → parse → EDA → **walk-forward backtest of five forecasting models
against naive baselines** → pick the winner per commodity (shipping the naive model if it wins —
that is the honest thing to do) → empirical prediction intervals from backtest errors →
export `prices.json`, which the PlotProof sell flow renders.

**Honesty contract:** every number shipped is measured. Forecast intervals are the 10th–90th
percentile of *actual* backtest errors, not theoretical bands. Reference prices are world/auction
prices, not farm-gate — the app says so on screen and uses them only to help a farmer judge offers.

**Runtime:** ~5–10 min on the free CPU runtime. Re-run monthly (the Pink Sheet updates at the
start of each month) and replace the JSON in the repo.


In [ ]:
import json, re, time, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(17)


In [ ]:
# World Bank Pink Sheet, monthly workbook. The World Bank rotates the document
# id roughly yearly and stops updating the old one, so we try newest-first and
# LOUDLY refuse stale data below. Current release page:
# https://www.worldbank.org/en/research/commodity-markets
URLS = [
    # 2026 release id (data through the current month)
    "https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/related/CMO-Historical-Data-Monthly.xlsx",
    # older 2021 id - STILL SERVED but frozen at 2024-12; fallback only
    "https://thedocs.worldbank.org/en/doc/5d903e848db1d1b83e0ec8f744e55570-0350012021/related/CMO-Historical-Data-Monthly.xlsx",
]
for url in URLS:
    try:
        urllib.request.urlretrieve(url, "cmo_monthly.xlsx")
        print("downloaded:", url.split("/doc/")[1].split("/")[0])
        break
    except Exception as e:
        print("failed:", e)
raw = pd.read_excel("cmo_monthly.xlsx", sheet_name="Monthly Prices", header=None)

# The sheet has a few metadata rows before the data. Find them instead of
# hard-coding offsets: data rows have dates like 1960M01 in column 0.
is_date = raw[0].astype(str).str.match(r"^\d{4}M\d{2}$")
start = int(np.argmax(is_date.to_numpy()))
hdr = None
for r in range(start - 1, max(start - 10, -1), -1):
    if raw.iloc[r].astype(str).str.contains("Cocoa", case=False).any():
        hdr = r
        break
assert hdr is not None, "could not locate the commodity-name header row"
names = raw.iloc[hdr].astype(str).str.strip()
units = raw.iloc[hdr + 1].astype(str).str.strip()

df = raw.iloc[start:].copy()
df.index = pd.PeriodIndex(df[0].astype(str).str.replace("M", "-"), freq="M")
df = df.drop(columns=[0]).apply(pd.to_numeric, errors="coerce")
df.columns = pd.MultiIndex.from_arrays([names[1:], units[1:]])
print(f"{df.shape[0]} months, {df.shape[1]} series, {df.index[0]} .. {df.index[-1]}")

# Freshness gate: a farmer must never be shown an old price as current. The app
# also refuses to render prices older than 3 months, but fail early and loudly.
months_old = (pd.Period.now("M") - df.index[-1]).n
if months_old > 3:
    raise SystemExit(
        f"REFUSING TO CONTINUE: data ends {df.index[-1]} ({months_old} months old). "
        "The World Bank has likely rotated the Pink Sheet URL again - get the current "
        "monthly xlsx link from https://www.worldbank.org/en/research/commodity-markets "
        "and put it first in URLS above.")
print(f"freshness OK: latest month {df.index[-1]} ({months_old} months old)")


In [ ]:
# The commodities PlotProof's catalog can use, matched defensively by name.
# key -> (Pink Sheet name to search, display label, app productIds it serves)
WANTED = {
    "tea_colombo":    ("Tea, Colombo",    "Tea (Colombo auction)",   ["black_tea"]),
    "coffee_arabica": ("Coffee, Arabica", "Coffee, Arabica",         ["coffee_green"]),
    "coffee_robusta": ("Coffee, Robusta", "Coffee, Robusta",         []),
    "rubber_rss3":    ("Rubber, RSS3",    "Natural rubber (RSS3)",   ["natural_rubber"]),
    "cocoa":          ("Cocoa",           "Cocoa beans",             ["cocoa_beans"]),
    "coconut_oil":    ("Coconut oil",     "Coconut oil (coconut-complex proxy)", ["desiccated_coconut"]),
}

def find_col(search):
    cands = [c for c in df.columns if str(c[0]).casefold().startswith(search.casefold())]
    if not cands:
        cands = [c for c in df.columns if search.split(",")[0].casefold() in str(c[0]).casefold()]
    assert cands, f"series not found: {search}"
    return cands[0]

series = {}
for key, (search, label, products) in WANTED.items():
    col = find_col(search)
    s = df[col].dropna().astype(float)
    unit = str(col[1])
    if "/mt" in unit.casefold():   # tonnes -> kg, the unit a farmer thinks in
        s, unit = s / 1000.0, "USD/kg"
    else:
        unit = "USD/kg"
    series[key] = {"s": s, "label": label, "unit": unit, "products": products,
                   "resolved": str(col[0])}
    print(f"{key:15s} <- '{col[0]}' ({col[1]}), {len(s)} months, latest {s.index[-1]} = {s.iloc[-1]:.2f} {unit}")


In [ ]:
# 25 years of each series. Eyeball check: do these look like real markets? (Cocoa's
# 2024 spike, rubber's 2011 peak, and covid dislocations should all be visible.)
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, (key, d) in zip(axes.flat, series.items()):
    s = d["s"][d["s"].index >= "2001-01"]
    ax.plot(s.index.to_timestamp(), s.values, lw=1)
    ax.set_title(f"{d['label']} ({d['unit']})", fontsize=10)
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Five forecasters, from dumbest to fanciest. If the dumb one wins the backtest,
# the dumb one ships - a fancier model that measures worse is marketing, not ML.
from sklearn.ensemble import HistGradientBoostingRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def fc_naive(s, h):
    return float(s.iloc[-1])

def fc_snaive(s, h):
    return float(s.iloc[-12 + (h - 1)]) if len(s) >= 12 else fc_naive(s, h)

def fc_drift(s, h):
    tail = s.iloc[-13:]
    slope = (tail.iloc[-1] - tail.iloc[0]) / (len(tail) - 1)
    return float(s.iloc[-1] + slope * h)

def fc_ets(s, h):
    try:
        m = ExponentialSmoothing(s.values, trend="add", damped_trend=True,
                                 seasonal="add", seasonal_periods=12).fit(optimized=True)
    except Exception:
        m = ExponentialSmoothing(s.values, trend="add", damped_trend=True).fit(optimized=True)
    return float(m.forecast(h)[-1])

LAGS = [1, 2, 3, 6, 12]

def _gbm_features(vals, i):
    return [vals[i - l] for l in LAGS] + [np.mean(vals[i - 3:i]), (i % 12) + 1]

def fc_gbm(s, h):
    vals = list(s.values)
    n = len(vals)
    Xtr = [_gbm_features(vals, i) for i in range(12, n)]
    ytr = vals[12:]
    m = HistGradientBoostingRegressor(max_iter=200, random_state=17)
    m.fit(Xtr, ytr)
    for _ in range(h):   # recursive multi-step
        vals.append(float(m.predict([_gbm_features(vals, len(vals))])[0]))
    return vals[-1]

MODELS = {"naive": fc_naive, "seasonal_naive": fc_snaive, "drift": fc_drift,
          "ETS_damped": fc_ets, "GBM_lags": fc_gbm}


In [ ]:
# Walk-forward backtest: 60 monthly origins, horizons 1-3. The model only ever
# sees data up to the origin - no leakage, refit at every origin.
N_ORIGINS, HORIZONS = 60, [1, 2, 3]
backtest = {}   # key -> model -> h -> list[(actual, pred)]
t0 = time.time()
for key, d in series.items():
    s = d["s"]
    res = {m: {h: [] for h in HORIZONS} for m in MODELS}
    for o in range(N_ORIGINS):
        cut = len(s) - HORIZONS[-1] - (N_ORIGINS - 1 - o)
        train = s.iloc[:cut]
        for name, fn in MODELS.items():
            for h in HORIZONS:
                actual = float(s.iloc[cut + h - 1])
                res[name][h].append((actual, fn(train, h)))
    backtest[key] = res
    print(f"{key} backtested ({time.time() - t0:.0f}s)")

def mape(pairs):
    return float(np.mean([abs(p - a) / abs(a) for a, p in pairs]) * 100)

rows = []
for key, res in backtest.items():
    for name in MODELS:
        rows.append({"commodity": key, "model": name,
                     **{f"MAPE_h{h}": round(mape(res[name][h]), 2) for h in HORIZONS}})
table = pd.DataFrame(rows)
table["MAPE_avg"] = table[[f"MAPE_h{h}" for h in HORIZONS]].mean(axis=1).round(2)
print(table.pivot(index="commodity", columns="model", values="MAPE_avg").to_string())

CHOSEN = {}
for key in series:
    sub = table[table.commodity == key]
    best = sub.loc[sub.MAPE_avg.idxmin()]
    CHOSEN[key] = best.model
    naive_avg = float(sub[sub.model == "naive"].MAPE_avg.iloc[0])
    print(f"{key:15s} -> {best.model:15s} (MAPE {best.MAPE_avg:.2f}% vs naive {naive_avg:.2f}%)")


## Reading the backtest honestly

Commodity prices are close to random walks at monthly horizons, so expect the naive model to be
**hard to beat** — single-digit improvements are normal, and for some commodities naive itself
wins. That result is a *feature* of this pipeline: the app ships whichever model measured best,
labels it, and shows the naive baseline next to it so nobody can dress up a random walk as AI.

What the farmer actually gets from this is mostly **not** the forecast — it is the *current
reference price, its 5-year context, and an uncertainty band that is honest about how much prices
move in a month*. The forecast is a small extra, priced at exactly its measured error.


In [ ]:
# Final forecasts + empirical intervals, then the JSON the app renders.
from datetime import datetime, timezone

def month_str(p):
    return f"{p.year:04d}-{p.month:02d}"

out = {
    "source": {"name": "World Bank Commodity Price Data (Pink Sheet)",
               "url": "https://www.worldbank.org/en/research/commodity-markets",
               "license": "CC BY 4.0"},
    "generatedAt": datetime.now(timezone.utc).isoformat(),
    "commodities": {},
    "productMap": {},
    "caveats": [
        "These are world reference prices (auction / FOB), not farm-gate offers. Use them to judge the offers you receive, not as a promised price.",
        "Coconut oil is a proxy for the coconut complex; no world series exists for desiccated coconut itself.",
        "Forecast bands are the 10th-90th percentile of real backtest errors over the last 60 months.",
        "No reference series exists for cinnamon, pepper, or cardamom; the app says so rather than inventing one.",
    ],
}

for key, d in series.items():
    s, name = d["s"], CHOSEN[key]
    fn = MODELS[name]
    res = backtest[key][name]
    fc = []
    for h in HORIZONS:
        point = fn(s, h)
        ratios = [a / p for a, p in res[h] if p > 0]
        lo, hi = np.quantile(ratios, 0.1), np.quantile(ratios, 0.9)
        month = month_str(s.index[-1] + h)
        fc.append({"month": month, "price": round(point, 3),
                   "low": round(point * lo, 3), "high": round(point * hi, 3)})
    last5y = s.iloc[-60:]
    pctile = float((last5y <= s.iloc[-1]).mean() * 100)
    yoy = float((s.iloc[-1] / s.iloc[-13] - 1) * 100) if len(s) > 13 else None
    out["commodities"][key] = {
        "label": d["label"],
        "unit": d["unit"],
        "sourceSeries": d["resolved"],
        "latest": {"month": month_str(s.index[-1]), "price": round(float(s.iloc[-1]), 3)},
        "history": [{"month": month_str(i), "price": round(float(v), 3)}
                    for i, v in s.iloc[-24:].items()],
        "forecast": fc,
        "context": {"pctile5y": round(pctile, 1),
                    "yoyChangePct": round(yoy, 1) if yoy is not None else None},
        "model": {"name": name,
                  "backtestMape1m": round(mape(res[1]), 2),
                  "naiveMape1m": round(mape(backtest[key]["naive"][1]), 2),
                  "windowMonths": N_ORIGINS},
    }
    for pid in d["products"]:
        out["productMap"][pid] = key

with open("prices.json", "w") as f:
    json.dump(out, f, indent=2)
print(json.dumps({k: v["latest"] for k, v in out["commodities"].items()}, indent=2))

try:
    from google.colab import files
    files.download("prices.json")
except ImportError:
    print("not in Colab - prices.json is in the working directory")


## What to do with `prices.json`

1. Commit it to the PlotProof repo at **`public/models/prices.json`** and deploy.
   The sell flow's results screen then shows each farmer their product's current world
   reference price, a 24-month sparkline, the value of *their* stated quantity at that
   price, and the honest forecast band. Until the file exists, the app shows nothing —
   it never invents a price.
2. **Re-run this notebook monthly** (the Pink Sheet updates at the start of each month)
   and replace the file. Later this can become a scheduled GitHub Action; the notebook
   is already deterministic end-to-end.
3. When you want cinnamon/pepper/cardamom: the Sri Lanka Export Development Board and
   Central Bank publish indicative spice prices, but only as PDFs/bulletins — digitising
   those honestly is a separate (worthwhile) project.
